In [1]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [2]:
ratings=pd.read_csv('ratings.csv')
movie=pd.read_csv('movies-new.csv')

In [3]:
def collaborative_filtering_recommend(user_id, ratings_df, movies_df, top_n):
    """User-based Collaborative Filtering"""
    # Tạo user-item matrix
    user_item_matrix = ratings_df.pivot(index='userId', columns='movieId', values='rating').fillna(0)
    
    # Tính user similarity
    user_similarity = cosine_similarity(user_item_matrix)
    
    # Tìm similar users
    user_idx = user_item_matrix.index.get_loc(user_id)
    similar_users = user_similarity[user_idx].argsort()[::-1][1:11]  # Top 10 similar users
    
    # Predict ratings
    predictions = []
    for movie_id in movies_df['movieId']:
        if movie_id in user_item_matrix.columns:
            # Tính weighted average rating
            similar_ratings = []
            similar_weights = []
            for similar_user_idx in similar_users:
                similar_user_id = user_item_matrix.index[similar_user_idx]
                rating = user_item_matrix.loc[similar_user_id, movie_id]
                if rating > 0:
                    similarity = user_similarity[user_idx, similar_user_idx]
                    similar_ratings.append(rating * similarity)
                    similar_weights.append(similarity)
            
            if similar_weights:
                predicted_rating = sum(similar_ratings) / sum(similar_weights)
                predictions.append((movie_id, predicted_rating))
    predictions.sort(key=lambda x: x[1], reverse=True)
     # return [movie_id for movie_id, _ in predictions[:top_n]]
    # Trả về cả thông tin phim
    top_movies = [(movie_id, predicted_rating) for movie_id, predicted_rating in predictions[:top_n]]
    watched_movies = set(ratings_df[ratings_df['userId'] == user_id]['movieId'])
    # return top_movies
    for movie_id, score in top_movies:
        if movie_id in user_item_matrix.columns and movie_id not in watched_movies:
            title = movies_df.loc[movies_df['movieId'] == movie_id, 'title'].values[0]
            print(movie_id,title, score)
    print(len(top_movies))

In [8]:
collaborative_filtering_recommend(1,ratings,movie,top_n=10)

514 Ref, The (1994) 5.0
529 Searching for Bobby Fischer (1993) 5.0
720 Wallace & Gromit: The Best of Aardman Animation (1996) 5.0
909 Apartment, The (1960) 5.0
915 Sabrina (1954) 5.0
1104 Streetcar Named Desire, A (1951) 5.0
1178 Paths of Glory (1957) 5.0
10


In [9]:
user_item_matrix=ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
print(user_item_matrix)

movieId  1       2       3       4       5       6       7       8       \
userId                                                                    
1           4.0     0.0     4.0     0.0     0.0     4.0     0.0     0.0   
2           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
5           4.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
...         ...     ...     ...     ...     ...     ...     ...     ...   
606         2.5     0.0     0.0     0.0     0.0     0.0     2.5     0.0   
607         4.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
608         2.5     2.0     2.0     0.0     0.0     0.0     0.0     0.0   
609         3.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
610         5.0     0.0     0.0     0.0     0.0     5.0     0.0     0.0   

movieId  9       10     

In [11]:
user_similarity=cosine_similarity(user_item_matrix)
print(user_similarity)

[[1.         0.02728287 0.05972026 ... 0.29109737 0.09357193 0.14532081]
 [0.02728287 1.         0.         ... 0.04621095 0.0275654  0.10242675]
 [0.05972026 0.         1.         ... 0.02112846 0.         0.03211875]
 ...
 [0.29109737 0.04621095 0.02112846 ... 1.         0.12199271 0.32205486]
 [0.09357193 0.0275654  0.         ... 0.12199271 1.         0.05322546]
 [0.14532081 0.10242675 0.03211875 ... 0.32205486 0.05322546 1.        ]]


In [12]:
user_idx = user_item_matrix.index.get_loc(1)
similar_users = user_similarity[user_idx].argsort()[::-1][1:11]
print(similar_users)

[265 312 367  56  90 468  38 287 451  44]


In [18]:
# Lấy index top 10 similar users
similar_users_idx = user_similarity[user_idx].argsort()[::-1][1:11]

# In ra giá trị similarity tương ứng
similar_users = user_similarity[user_idx][similar_users_idx]
print(similar_users)


[0.35740771 0.35156152 0.34512705 0.34503428 0.33472692 0.33066432
 0.32978223 0.32969953 0.32804834 0.32792169]
